# Life Science Notebook: Clinical Response and Safety Modeling (Expanded)

This notebook simulates a Phase II/III-style patient cohort and builds review-ready efficacy and safety analytics.

Quality goals:
- transparent assumptions and reproducible simulation,
- dual-endpoint modeling (efficacy + severe adverse events),
- robust evaluation (discrimination + calibration),
- subgroup consistency checks,
- benefit-risk dose policy recommendation.

## 0) Imports and setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

np.random.seed(123)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## 1) Simulate clinical dataset

In [ ]:
n = 5200

trial = pd.DataFrame(
    {
        "age": np.random.randint(18, 86, size=n),
        "sex": np.random.choice(["F", "M"], size=n),
        "baseline_biomarker": np.random.normal(0.95, 0.38, size=n),
        "comorbidity_index": np.random.poisson(1.3, size=n),
        "baseline_severity": np.clip(np.random.normal(5.2, 1.6, size=n), 0.2, 10.0),
        "egfr": np.clip(np.random.normal(82, 18, size=n), 20, 140),
        "dose_mg": np.random.choice([25, 50, 100, 150], p=[0.20, 0.35, 0.30, 0.15], size=n),
        "site_region": np.random.choice(["NA", "EU", "APAC", "LATAM"], p=[0.40, 0.33, 0.18, 0.09], size=n),
    }
)

trial["elderly"] = (trial["age"] >= 65).astype(int)

# Latent efficacy signal
resp_logit = (
    -1.00
    + 0.011 * trial["dose_mg"].to_numpy()
    + 0.85 * trial["baseline_biomarker"].to_numpy()
    - 0.016 * trial["age"].to_numpy()
    - 0.19 * trial["comorbidity_index"].to_numpy()
    - 0.08 * trial["baseline_severity"].to_numpy()
    + 0.003 * trial["egfr"].to_numpy()
    + np.where(trial["site_region"].to_numpy() == "EU", 0.06, 0.0)
)

# Latent severe AE signal
ae_logit = (
    -2.15
    + 0.010 * trial["dose_mg"].to_numpy()
    + 0.018 * trial["age"].to_numpy()
    + 0.33 * trial["comorbidity_index"].to_numpy()
    + 0.09 * trial["baseline_severity"].to_numpy()
    - 0.004 * trial["egfr"].to_numpy()
    + np.where(trial["sex"].to_numpy() == "F", 0.05, 0.0)
)

trial["efficacy_response"] = np.random.binomial(1, 1 / (1 + np.exp(-resp_logit)), size=n)
trial["grade3_ae"] = np.random.binomial(1, 1 / (1 + np.exp(-ae_logit)), size=n)

trial.head()

In [ ]:
print(f"Patients: {len(trial):,}")
print(f"Efficacy response rate: {trial['efficacy_response'].mean():.2%}")
print(f"Grade >=3 AE rate: {trial['grade3_ae'].mean():.2%}")

## 2) Dose-level efficacy and safety profile

In [ ]:
profile = trial.groupby("dose_mg", as_index=False).agg(
    response_rate=("efficacy_response", "mean"),
    severe_ae_rate=("grade3_ae", "mean"),
    n_patients=("dose_mg", "size"),
)
profile

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].plot(profile["dose_mg"], profile["response_rate"], marker="o", color="#4C72B0")
ax[0].set_title("Dose vs Response Rate")
ax[0].set_xlabel("Dose (mg)")
ax[0].set_ylabel("Response rate")

ax[1].plot(profile["dose_mg"], profile["severe_ae_rate"], marker="o", color="#C44E52")
ax[1].set_title("Dose vs Severe AE Rate")
ax[1].set_xlabel("Dose (mg)")
ax[1].set_ylabel("Severe AE rate")

plt.tight_layout()
plt.show()

## 3) Prepare features and splits

In [ ]:
features = [
    "age",
    "sex",
    "baseline_biomarker",
    "comorbidity_index",
    "baseline_severity",
    "egfr",
    "dose_mg",
    "site_region",
    "elderly",
]

num_cols = [
    "age",
    "baseline_biomarker",
    "comorbidity_index",
    "baseline_severity",
    "egfr",
    "dose_mg",
    "elderly",
]
cat_cols = ["sex", "site_region"]

prep = ColumnTransformer(
    [
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

X = trial[features]
y_eff = trial["efficacy_response"]
y_ae = trial["grade3_ae"]

X_train_eff, X_test_eff, y_train_eff, y_test_eff = train_test_split(
    X, y_eff, test_size=0.30, random_state=123, stratify=y_eff
)

X_train_ae, X_test_ae, y_train_ae, y_test_ae = train_test_split(
    X, y_ae, test_size=0.30, random_state=123, stratify=y_ae
)

## 4) Model benchmarking for efficacy and safety

In [ ]:
def build_models():
    logit = Pipeline(
        [
            ("prep", prep),
            ("clf", LogisticRegression(max_iter=500, random_state=123)),
        ]
    )
    rf = Pipeline(
        [
            ("prep", prep),
            (
                "clf",
                RandomForestClassifier(
                    n_estimators=320,
                    max_depth=9,
                    min_samples_leaf=12,
                    random_state=123,
                    n_jobs=-1,
                ),
            ),
        ]
    )
    return {"Logistic": logit, "RandomForest": rf}


def evaluate_models(X_train, X_test, y_train, y_test, endpoint_name: str):
    rows = []
    preds = {}
    for name, model in build_models().items():
        model.fit(X_train, y_train)
        p = model.predict_proba(X_test)[:, 1]
        preds[name] = (model, p)
        rows.append(
            {
                "endpoint": endpoint_name,
                "model": name,
                "roc_auc": roc_auc_score(y_test, p),
                "pr_auc": average_precision_score(y_test, p),
                "brier": brier_score_loss(y_test, p),
            }
        )
    return pd.DataFrame(rows).sort_values("roc_auc", ascending=False), preds

res_eff, preds_eff = evaluate_models(X_train_eff, X_test_eff, y_train_eff, y_test_eff, "efficacy_response")
res_ae, preds_ae = evaluate_models(X_train_ae, X_test_ae, y_train_ae, y_test_ae, "grade3_ae")

benchmark = pd.concat([res_eff, res_ae], ignore_index=True)
benchmark

In [ ]:
plt.figure(figsize=(8.8, 4.5))
plot_df = benchmark.melt(id_vars=["endpoint", "model"], value_vars=["roc_auc", "pr_auc", "brier"], var_name="metric")
sns.barplot(data=plot_df, x="metric", y="value", hue="model", palette="Set2")
plt.title("Model Benchmark Across Endpoints")
plt.tight_layout()
plt.show()

## 5) Select champion models and assess calibration

In [ ]:
champ_eff_name = res_eff.iloc[0]["model"]
champ_ae_name = res_ae.iloc[0]["model"]

model_eff, p_eff = preds_eff[champ_eff_name]
model_ae, p_ae = preds_ae[champ_ae_name]

print(f"Efficacy champion: {champ_eff_name}")
print(f"Safety champion  : {champ_ae_name}")

frac_eff, mean_eff = calibration_curve(y_test_eff, p_eff, n_bins=12)
frac_ae, mean_ae = calibration_curve(y_test_ae, p_ae, n_bins=12)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.8))

ax[0].plot(mean_eff, frac_eff, marker="o", color="#4C72B0", label="Efficacy")
ax[0].plot([0, 1], [0, 1], linestyle="--", color="black")
ax[0].set_title("Efficacy Calibration")
ax[0].set_xlabel("Predicted probability")
ax[0].set_ylabel("Observed rate")

ax[1].plot(mean_ae, frac_ae, marker="o", color="#C44E52", label="Severe AE")
ax[1].plot([0, 1], [0, 1], linestyle="--", color="black")
ax[1].set_title("Safety Calibration")
ax[1].set_xlabel("Predicted probability")
ax[1].set_ylabel("Observed rate")

plt.tight_layout()
plt.show()

## 6) Patient-level benefit-risk scoring

In [ ]:
trial_scored = trial.copy()
trial_scored["p_response"] = model_eff.predict_proba(trial_scored[features])[:, 1]
trial_scored["p_severe_ae"] = model_ae.predict_proba(trial_scored[features])[:, 1]

# Utility definition (domain-tunable): efficacy positive, severe AE negative
ae_penalty = 0.85
trial_scored["benefit_risk_score"] = trial_scored["p_response"] - ae_penalty * trial_scored["p_severe_ae"]

trial_scored[["p_response", "p_severe_ae", "benefit_risk_score"]].describe().T

In [ ]:
dose_policy = trial_scored.groupby("dose_mg", as_index=False).agg(
    mean_benefit_risk=("benefit_risk_score", "mean"),
    mean_response_prob=("p_response", "mean"),
    mean_ae_prob=("p_severe_ae", "mean"),
    observed_response_rate=("efficacy_response", "mean"),
    observed_severe_ae_rate=("grade3_ae", "mean"),
    n=("dose_mg", "size"),
)

dose_policy = dose_policy.sort_values("mean_benefit_risk", ascending=False)
dose_policy

In [ ]:
best_dose = int(dose_policy.iloc[0]["dose_mg"])

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))

sns.barplot(data=dose_policy.sort_values("dose_mg"), x="dose_mg", y="mean_benefit_risk", ax=ax[0], palette="crest")
ax[0].set_title("Dose Ranking by Benefit-Risk")
ax[0].set_xlabel("Dose (mg)")

tmp = dose_policy.sort_values("dose_mg")
ax[1].plot(tmp["dose_mg"], tmp["mean_response_prob"], marker="o", label="Predicted response", color="#4C72B0")
ax[1].plot(tmp["dose_mg"], tmp["mean_ae_prob"], marker="o", label="Predicted severe AE", color="#C44E52")
ax[1].set_title("Predicted Efficacy-Safety Frontier")
ax[1].set_xlabel("Dose (mg)")
ax[1].legend()

plt.tight_layout()
plt.show()

print(f"Top dose by mean benefit-risk score: {best_dose} mg")

## 7) Subgroup consistency checks

In [ ]:
def subgroup_perf(df: pd.DataFrame, group_col: str, min_n: int = 120):
    out = []
    for grp, sub in df.groupby(group_col):
        if len(sub) < min_n:
            continue
        out.append(
            {
                "group_col": group_col,
                "group": grp,
                "n": len(sub),
                "obs_response": sub["efficacy_response"].mean(),
                "pred_response": sub["p_response"].mean(),
                "obs_severe_ae": sub["grade3_ae"].mean(),
                "pred_severe_ae": sub["p_severe_ae"].mean(),
                "mean_benefit_risk": sub["benefit_risk_score"].mean(),
            }
        )
    return pd.DataFrame(out)

subgroup_table = pd.concat(
    [
        subgroup_perf(trial_scored, "sex"),
        subgroup_perf(trial_scored, "site_region"),
        subgroup_perf(trial_scored, "elderly"),
    ],
    ignore_index=True,
)

subgroup_table

In [ ]:
plot_sub = subgroup_table[subgroup_table["group_col"] == "site_region"].copy()
plt.figure(figsize=(8, 4.5))
sns.barplot(data=plot_sub, x="group", y="mean_benefit_risk", palette="viridis")
plt.title("Benefit-Risk Score by Site Region")
plt.xlabel("Region")
plt.ylabel("Mean benefit-risk score")
plt.tight_layout()
plt.show()

## 8) Decision sensitivity to safety penalty

In [ ]:
penalties = np.linspace(0.40, 1.20, 17)
rows = []

for pen in penalties:
    tmp = trial_scored.copy()
    tmp["score"] = tmp["p_response"] - pen * tmp["p_severe_ae"]
    by_dose = tmp.groupby("dose_mg", as_index=False)["score"].mean()
    best_row = by_dose.sort_values("score", ascending=False).iloc[0]
    rows.append({"ae_penalty": pen, "best_dose": int(best_row["dose_mg"]), "best_score": best_row["score"]})

sens = pd.DataFrame(rows)
sens.head()

In [ ]:
plt.figure(figsize=(8, 4.5))
sns.lineplot(data=sens, x="ae_penalty", y="best_dose", marker="o", color="#8172B2")
plt.title("Best Dose vs Safety Penalty")
plt.xlabel("Safety penalty weight")
plt.ylabel("Recommended dose (mg)")
plt.tight_layout()
plt.show()

## 9) Final summary

- The notebook now supports review-grade efficacy/safety decision analysis with explicit tradeoffs.
- Calibration and subgroup checks complement core discrimination metrics.
- Dose recommendations are stable only within a chosen benefit-risk preference; sensitivity analysis makes that explicit.